# Meeting Minutes Generator

### ==========================================================
### FINAL 
### MP3 -> Transcript (Whisper) -> High-quality Minutes (Ollama)
### ==========================================================

#### ---- Import Section ----

In [1]:
import torch                         # torch = PyTorch library (used here for GPU/CPU detection + dtype like float16/float32)
import librosa                       # librosa = audio loading/processing library (loads MP3 into arrays)
from transformers import pipeline    # pipeline = easy Hugging Face wrapper (we use it for Whisper ASR transcription)
from openai import OpenAI            # OpenAI client (we use it with Ollama's OpenAI-compatible API)
from IPython.display import display, Markdown  # display/Markdown = show Markdown output nicely inside Jupyter Notebook

#### -----------------------------
#### 0) SETTINGS (change only these)
#### -----------------------------


In [ ]:
# audio_file_path:
# - "r" before the string means "raw string"
# - raw string avoids issues with backslashes in Windows paths (\)
audio_file_path = r"D:\work\llm_engineering\Data\denver_extract.mp3"  # path to your input MP3 file

# whisper_model_name:
# - Whisper model for speech-to-text (ASR)
# - base.en is a good balance; tiny.en is faster but less accurate
whisper_model_name = "openai/whisper-base.en"

# minutes_model_name:
# - the LLM you will call via Ollama
minutes_model_name = "gpt-oss:120b-cloud"

# max_transcript_characters:
# - limits transcript length to keep LLM generation stable and faster
max_transcript_characters = 14000

# chunk_length_seconds:
# - Whisper will process audio in chunks of this many seconds
# - helps with long audio and memory usage
chunk_length_seconds = 45

#### ==========================================================
#### 1) GPU or CPU?
#### ==========================================================

In [3]:
# is_gpu_available:
# - torch.cuda.is_available() returns True if CUDA GPU is available, else False
is_gpu_available = torch.cuda.is_available()

# device_number:
# - transformers pipeline uses:
#   device=0  -> first GPU (cuda:0)
#   device=-1 -> CPU
device_number = 0 if is_gpu_available else -1  # "if GPU then 0 else -1"

# Print whether CUDA is available
print("CUDA available:", is_gpu_available)

# If GPU exists, print GPU name (example: "NVIDIA GeForce GTX 1650")
if is_gpu_available:
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA GeForce GTX 1650


#### ==========================================================
#### 2) Load audio (Whisper expects 16kHz mono)
#### ==========================================================

In [4]:
print("\n[1/3] Loading audio...")  # \n makes a blank line before the text

# librosa.load(...) returns TWO things:
# 1) audio_data = waveform array (numbers representing sound amplitude)
# 2) sample_rate = samples per second
#
# We force:
# - sr=16000   -> resample audio to 16kHz (Whisper expects 16kHz)
# - mono=True  -> convert to mono (single channel)
audio_data, sample_rate = librosa.load(audio_file_path, sr=16000, mono=True)

# len(audio_data) = number of samples
# sample_rate = samples per second
# len(audio_data)/sample_rate = total seconds
print("Audio length (sec):", round(len(audio_data) / sample_rate, 1))  # round(..., 1) = 1 decimal place


[1/3] Loading audio...
Audio length (sec): 900.0


#### ==========================================================
#### 3) Transcribe with Whisper (with timestamps)
#### ==========================================================

In [5]:
print("[2/3] Transcribing with Whisper...")

# speech_to_text_model = pipeline(...):
# - task: "automatic-speech-recognition" = speech to text
# - model: whisper_model_name
# - device: GPU/CPU selection
# - torch_dtype:
#     float16 on GPU (faster + less memory)
#     float32 on CPU (safe/default)
# - return_timestamps=True:
#     asks Whisper to return timestamps for parts of transcript
# - chunk_length_s:
#     audio is processed in segments
# - stride_length_s=(2,2):
#     overlap between chunks (left and right) to reduce cut-off words
speech_to_text_model = pipeline(
    "automatic-speech-recognition",              # task name
    model=whisper_model_name,                    # whisper model ID
    device=device_number,                        # 0 for GPU, -1 for CPU
    torch_dtype=torch.float16 if is_gpu_available else torch.float32,  # pick dtype based on GPU availability
    return_timestamps=True,                      # get timestamps in output
    chunk_length_s=chunk_length_seconds,         # chunk size in seconds
    stride_length_s=(2, 2),                      # overlap in seconds (left, right)
)

# transcription_result:
# - calling the pipeline like a function runs ASR
# - input is audio_data (waveform array)
transcription_result = speech_to_text_model(audio_data)

# convert_seconds_to_mmss(seconds_value):
# - converts time in seconds to "MM:SS"
# - handles tuple/list timestamps too (sometimes timestamp is (start,end))
def convert_seconds_to_mmss(seconds_value):
    """Convert seconds -> MM:SS (handles tuple timestamps too)."""

    # If timestamp is like (start, end), take the start time
    if isinstance(seconds_value, (tuple, list)):  # isinstance checks the type
        seconds_value = seconds_value[0]

    # If timestamp is missing (None), return placeholder
    if seconds_value is None:
        return "??:??"

    # Convert to integer seconds (remove decimals)
    seconds_value = int(seconds_value)

    # divmod(a, b) returns (a//b, a%b)
    # Here: minutes = seconds_value//60, seconds = seconds_value%60
    minutes, seconds = divmod(seconds_value, 60)

    # f-string formatting:
    # {minutes:02d} means "2 digits, pad with 0"
    return f"{minutes:02d}:{seconds:02d}"


# Build timestamped transcript (one line per chunk)
# Whisper pipeline can return output as:
# - transcription_result["chunks"] : list of chunk objects with "text" and "timestamp"
# OR
# - transcription_result["text"]   : plain text string (no chunks)
if "chunks" in transcription_result and transcription_result["chunks"]:
    transcript_lines = []  # list to store each transcript line
    previous_line = ""     # used to avoid adding duplicate lines

    # Loop over each chunk produced by Whisper
    for chunk in transcription_result["chunks"]:
        # chunk.get("text") returns the chunk text if it exists, else None
        # ( ... or "" ) ensures we never call .strip() on None
        text_part = (chunk.get("text") or "").strip()  # strip removes leading/trailing whitespace

        # If chunk has no text, skip it
        if not text_part:
            continue

        # chunk.get("timestamp") gives time info
        # convert_seconds_to_mmss turns it into [MM:SS]
        formatted_line = f"[{convert_seconds_to_mmss(chunk.get('timestamp'))}] {text_part}"

        # Avoid repeating the exact same line twice
        if formatted_line != previous_line:
            transcript_lines.append(formatted_line)  # add line to list
            previous_line = formatted_line           # update last seen line

    # Join all lines into one big transcript string separated by newline
    full_transcript = "\n".join(transcript_lines)

else:
    # If no chunks exist, just use the normal "text" field
    full_transcript = transcription_result.get("text", "")

# Trim transcript to max_transcript_characters:
# - keeps it from being too long
# - helps speed and reduces model confusion
full_transcript = full_transcript[:max_transcript_characters]

# Show how many characters we will send to LLM
print("Transcription done. Characters used:", len(full_transcript))

[2/3] Transcribing with Whisper...


`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


Transcription done. Characters used: 9377


#### ==========================================================
#### 4) Generate Meeting Minutes via Ollama (PRO Prompt, no prints)
#### ==========================================================

In [6]:

print("[3/3] Generating meeting minutes via Ollama...\n")

# ollama_client:
# - OpenAI client object
# - base_url points to local Ollama OpenAI-compatible server
# - api_key is required by client but Ollama ignores it (placeholder)
ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  # placeholder for Ollama
)

# system_prompt:
# - system message tells the model "who it is" and strict rules
# - this heavily controls formatting + quality
system_prompt = """
You are an expert corporate secretary and council/board meeting scribe.

Your job:
Turn the transcript into clear, accurate, professional meeting minutes.

Output rules (STRICT):
- Output ONLY Markdown minutes (no preface, no explanation, no transcript)
- Do NOT copy transcript lines verbatim; rewrite and summarize (3-4 sentences)
- Keep facts faithful; do NOT invent details
- If something is unclear/missing, write 'Not specified'
- If no action items are explicitly assigned, write 'None mentioned'
- Use blank lines between sections
- Prefer concise bullets; use short paragraphs only where requested

Quality requirements:
- Write a descriptive Meeting Summary (6–8 sentences) explaining context, purpose, and outcomes
- Agenda should be inferred as best as possible from transcript
- Key Discussion Points should be grouped by topic, each topics should have 2–4 bullet points explanation.
- Each bullet should include a timestamp like [MM:SS] when relevant
- Decisions must be explicit; if implied but not confirmed, put under Key Discussion Points instead

Use EXACT headings and order:

# Meeting Summary
# Attendees
# Agenda
# Key Discussion Points
# Decisions Made
# Action Items
# Votes / Motions
# Risks / Blockers
# Next Steps

Attendees:
- Only include names/roles that are clearly mentioned; otherwise write 'Not specified'

Votes / Motions:
- If none clearly stated, write 'None mentioned'
""".strip()  # .strip() removes extra whitespace/newlines at start/end

# user_prompt:
# - includes the transcript and tells the model what to do
# - f-string inserts full_transcript into the message
user_prompt = f"""
Transcript (with timestamps):
{full_transcript}

Now generate the meeting minutes.
""".strip()

# response_stream:
# - chat.completions.create(...) sends messages to the model
# - model=minutes_model_name chooses which Ollama model to use
# - temperature=0 makes output more deterministic (less random)
# - stream=True returns tokens gradually (streaming)
response_stream = ollama_client.chat.completions.create(
    model=minutes_model_name,
    messages=[
        {"role": "system", "content": system_prompt},  # system instructions
        {"role": "user", "content": user_prompt},      # user transcript + request
    ],
    temperature=0,  # low randomness
    stream=True     # stream tokens (we still collect silently)
)

# Collect tokens silently (no console streaming)
generated_minutes = ""  # will store the final minutes text

# Loop through each streamed chunk from the model
for response_chunk in response_stream:
    # response_chunk.choices[0].delta contains the incremental token
    token_data = response_chunk.choices[0].delta

    # getattr(token_data, "content", None):
    # - safely get token_data.content if it exists
    # - otherwise return None
    token_text = getattr(token_data, "content", None)

    # Only append if this chunk contains actual text
    if token_text:
        generated_minutes += token_text  # add token text to final output string

[3/3] Generating meeting minutes via Ollama...



#### ==========================================================
#### 5) Display Nicely + Save to .md file
#### ==========================================================

In [7]:
# Display Markdown nicely in Jupyter Notebook
display(Markdown(generated_minutes))

# Save to a Markdown file in the current working directory
# with open(...) as file:
# - opens file for writing ("w")
# - encoding="utf-8" supports all characters safely

'''

with open("meeting_minutes.md", "w", encoding="utf-8") as file:
    file.write(generated_minutes)  # write the generated text into the file

print("Saved as: meeting_minutes.md")  # confirm saved file name

'''


# Meeting Summary
The City and County of Denver Council convened to approve the prior meeting’s minutes, share community announcements, and officially recognize Indigenous Peoples Day through a formal proclamation. Council members highlighted the symbolic “Confluence Week” logo, emphasizing water and cultural heritage. Councilman Clark promoted the inaugural Broadway Halloween Parade slated for October 8. Councilman Lopez read the Indigenous Peoples Day proclamation, underscoring respect for Native cultures and the upcoming National Indigenous Youth Leadership Conference. Members expressed appreciation for Indigenous art, cultural preservation, and the importance of inclusivity. No contentious issues arose, and the council affirmed its support for upcoming cultural events and initiatives.

# Attendees
- Councilman Lopez  
- Councilman Clark  
- Madam Secretary Raulkaw (Secretary)  
- Not specified (other council members referenced but unnamed)

# Agenda
- Approval of October 2 minutes  
- Council announcements (Halloween parade)  
- Presentation of Indigenous Peoples Day proclamation  
- Discussion of “Confluence Week” logo symbolism  
- Community and cultural remarks

# Key Discussion Points
- **Minutes Approval** – No corrections were offered; the October 2 minutes were stood to be approved. [00:00‑00:30]  
- **Halloween Parade Announcement** – Councilman Clark invited members to the first Broadway Halloween Parade on October 8, describing activities and encouraging attendance on October 21 at 6 p.m. [02:53‑03:07]  
- **Proclamation Reading** – Councilman Lopez read Proclamation 1127 (2017) honoring Indigenous Peoples Day, detailing Colorado’s tribal heritage and the upcoming National Indigenous Youth Leadership Conference. [06:17‑06:33]  
- **Cultural Significance of Logo** – Members praised the new logo’s water theme and its representation of cultural convergence, noting the importance of art, music, and land in preserving Indigenous identity. [13:01‑13:22]  
- **Community Reflections** – Several remarks highlighted pride in cultural heritage, the need for inclusivity, and the protection of sacred lands amid environmental concerns. [08:57‑14:44]

# Decisions Made
- October 2 minutes approved (no objections recorded).  
- Proclamation for Indigenous Peoples Day adopted and to be sealed and distributed.  

# Action Items
- **Councilman Clark** – Attend and promote the Broadway Halloween Parade on October 8. (No specific deadline)  
- **Secretary (Madam Secretary Raulkaw)** – Affix the city seal to the proclamation and circulate copies to the Denver American Indian Commission, Denver Public Schools, and the Colorado Commission on Indian Affairs. [06:33]  
- **Council members** – Continue support for the National Indigenous Youth Leadership Conference and related cultural initiatives. (Implicit, no formal assignment)

# Votes / Motions
None mentioned

# Risks / Blockers
None specified

# Next Steps
- Disseminate the sealed proclamation to relevant agencies and commissions.  
- Finalize logistical details for the Broadway Halloween Parade and ensure community outreach.  
- Follow up on planning for the National Indigenous Youth Leadership Conference and related Indigenous cultural events.  

'\n\nwith open("meeting_minutes.md", "w", encoding="utf-8") as file:\n    file.write(generated_minutes)  # write the generated text into the file\n\nprint("Saved as: meeting_minutes.md")  # confirm saved file name\n\n'